# Methods Tutorial: Word Error Rate (WER)

## Introduction

This tutorial teaches how to calculate **Word Error Rate (WER)** — the standard metric for evaluating the accuracy of speech recognition (ASR) systems.

Everything you need is contained in this notebook. You do not need any external project files.

### What You'll Learn

By the end of this tutorial, you will understand:

- What WER is and why it is the standard metric for ASR evaluation
- The concepts of **substitutions**, **insertions**, and **deletions** in word alignment
- How the WER formula works and how to interpret the result
- Why **text normalization** (lowercasing, punctuation removal) is standard practice before computing WER
- How to implement WER calculation in Python using the `jiwer` library
- How to handle edge cases (empty reference, empty hypothesis)
- How to visualize WER across a batch of transcriptions

## Tutorial Layout

| Part | Section | Topic |
|------|---------|-------|
| **Opening** | Introduction | Goals, scope, and what you will learn |
| **Opening** | Overview | What WER measures and the pipeline |
| **Background** | A | What is Word Error Rate? |
| **Background** | B | Edit operations — Substitutions, Insertions, Deletions |
| **Background** | C | Text normalization for fair comparison |
| **Setup** | — | Install from `requirements.txt` |
| **Setup** | 1 | Import libraries |
| **Setup** | 2 | Define functions |
| **Setup** | 3 | Create demo data |
| **Implementation** | 4 | Step 1 — Normalize text |
| **Implementation** | 5 | Step 2 — Calculate WER for a single pair |
| **Implementation** | 6 | Step 3 — Interpret the WER score |
| **Implementation** | 7 | Step 4 — Handle edge cases |
| **Implementation** | 8 | Step 5 — Batch WER over multiple transcriptions |
| **Visualization** | 9 | WER distribution across a batch |
| **Summary** | 10 | Flow summary and key point |
| **Conclusion** | 11 | What you learned and next steps |

## Overview

When an ASR system transcribes audio, its output (the **hypothesis**) is compared against a known correct transcript (the **reference**). WER quantifies how many words the system got wrong.

This notebook implements the evaluation pipeline:

1. Normalize both reference and hypothesis text (lowercase, remove punctuation)
2. Align words and count errors (substitutions, insertions, deletions)
3. Compute WER = errors / total reference words
4. Interpret the score (0% = perfect, 100% = completely wrong)

## A. What Is Word Error Rate?

**Word Error Rate (WER)** is the standard metric used by researchers and industry to evaluate the accuracy of automatic speech recognition systems.

WER measures the **minimum number of word-level edits** needed to transform the hypothesis (what the ASR system produced) into the reference (the correct transcript), divided by the total number of words in the reference.

### The WER formula

$$\text{WER} = \frac{S + I + D}{N}$$

Where:
- **S** = number of **substitutions** (words that were replaced with a wrong word)
- **I** = number of **insertions** (extra words the system added that aren't in the reference)
- **D** = number of **deletions** (words in the reference that the system missed entirely)
- **N** = total number of words in the **reference** transcript

### How to read WER

| WER | Interpretation |
|-----|----------------|
| 0.00 (0%) | Perfect transcription — every word matches |
| 0.05 (5%) | Excellent — nearly all words correct |
| 0.10–0.20 (10–20%) | Good — minor errors, still usable |
| 0.30–0.50 (30–50%) | Moderate — noticeable errors |
| 1.00 (100%) | Every word is wrong (or reference length = number of errors) |
| >1.00 (>100%) | Possible when many extra words are inserted |

**Lower is better.** A WER of 0.1668 means the system correctly transcribed about 83% of words.

## B. Edit Operations — Substitutions, Insertions, Deletions

WER is based on the **minimum edit distance** at the word level. The three types of errors are:

### Substitution (S)

A word in the reference was replaced by a **different** word in the hypothesis.

```
Reference:  I have sharp pain
Hypothesis: I have shark pain
                   ^^^^^
                   substitution: "sharp" → "shark"
```

### Insertion (I)

The hypothesis contains an **extra** word that does not appear in the reference.

```
Reference:  I have sharp pain
Hypothesis: I have a sharp pain
                   ^
                   insertion: "a" added
```

### Deletion (D)

A word in the reference is **missing** from the hypothesis.

```
Reference:  I have sharp pain in my chest
Hypothesis: I have sharp pain in chest
                                 ^^
                                 deletion: "my" missing
```

### Combined example

```
Reference:  "I have sharp pain"   (4 words, N=4)
Hypothesis: "I have a shark pain" (5 words)

Errors:
  1 substitution: "sharp" → "shark"
  1 insertion:    "a" added
  0 deletions

WER = (1 + 1 + 0) / 4 = 2/4 = 0.50 (50%)
```

## C. Text Normalization for Fair Comparison

Before computing WER, both the reference and hypothesis are **normalized**. This is standard practice in ASR evaluation, used by researchers and industry.

### Why normalize?

Without normalization, trivial differences would inflate the error rate:

| Reference | Hypothesis | Without normalization | With normalization |
|-----------|------------|----------------------|--------------------|
| "My shoulder hurts." | "my shoulder hurts" | WER > 0 (case + period differ) | WER = 0 (identical) |
| "Pain, sharp" | "pain sharp" | WER > 0 (comma differs) | WER = 0 (identical) |

### What normalization does

1. **Lowercase** all text — "My" and "my" become the same word
2. **Remove punctuation** — periods, commas, apostrophes, etc. are stripped

This ensures WER measures **actual word accuracy**, not formatting differences.

```
Before: "My muscle, in my shoulder, burns."  →  After: "my muscle in my shoulder burns"
Before: "my muscle in my shoulder burns"     →  After: "my muscle in my shoulder burns"

WER = 0.00 (perfect match after normalization)
```

## Dependencies

Install the required libraries from `requirements.txt` in this folder before running the notebook. Run the cell below:

Packages used: `jiwer` (WER computation), `matplotlib` (visualization).

In [ ]:
!pip install -r requirements.txt

## 1. Import Libraries

In [ ]:
import string                        # for punctuation removal during normalization
from typing import List, Tuple

import matplotlib.pyplot as plt
from jiwer import wer                # industry-standard WER implementation

## 2. Define Functions

These functions implement the WER calculation pipeline.

In [ ]:
def normalize_text(text: str) -> str:
    """Normalize text by lowercasing and removing punctuation.

    This is standard practice in ASR evaluation so that WER
    measures word accuracy, not formatting differences.
    """
    # str.maketrans maps every punctuation char to None, effectively deleting them
    return text.lower().translate(str.maketrans("", "", string.punctuation))


def calculate_wer(reference: str, hypothesis: str, normalize: bool = True) -> float:
    """Calculate Word Error Rate between reference and hypothesis text.

    WER = (Substitutions + Insertions + Deletions) / Total words in reference

    Args:
        reference: Ground truth transcription.
        hypothesis: Model-generated transcription.
        normalize: If True, lowercase and remove punctuation before comparison.

    Returns:
        WER value (0.0 = perfect match, higher = more errors).
    """
    # Edge case: empty ref with non-empty hyp → 1.0 (total failure);
    #            both empty → 0.0 (nothing to get wrong)
    if not reference.strip():
        return 1.0 if hypothesis.strip() else 0.0

    if normalize:
        reference = normalize_text(reference)
        hypothesis = normalize_text(hypothesis)

    return wer(reference, hypothesis)


def batch_wer(
    pairs: List[Tuple[str, str]],
    normalize: bool = True,
) -> List[float]:
    """Calculate WER for a list of (reference, hypothesis) pairs."""
    # One WER score per pair; useful for per-file analysis across a dataset
    return [calculate_wer(ref, hyp, normalize) for ref, hyp in pairs]


def get_demo_pairs() -> List[Tuple[str, str]]:
    """Return demo (reference, hypothesis) pairs for this tutorial."""
    # Each pair simulates a different ASR error type:
    #   pair 0: case-only diff (perfect after normalization)
    #   pair 1: substitution + insertion
    #   pair 2: case-only diff (perfect after normalization)
    #   pair 3: deletion ("my" missing)
    #   pair 4: deletion ("left" missing)
    #   pair 5: case-only diff (perfect after normalization)
    #   pair 6: substitution ("walk" → "walking")
    #   pair 7: substitution ("dizzy" → "busy")
    return [
        ("My muscle in my shoulder burns when I move my arm.",
         "my muscle in my shoulder burns when i move my arm"),
        ("I have sharp pain in my lower back.",
         "I have a shark pain in my lower back"),
        ("My head has been hurting for three days.",
         "my head has been hurting for three days"),
        ("I feel a burning sensation in my chest.",
         "i feel a burning sensation in chest"),
        ("There is numbness in my left hand.",
         "there is numbness in left hand"),
        ("I have been coughing for a week.",
         "i have been coughing for a week"),
        ("My knee swells up after I walk.",
         "my knee swells up after walking"),
        ("I get dizzy when I stand up quickly.",
         "i get busy when i stand up quickly"),
    ]


def plot_wer_distribution(
    labels: List[str],
    scores: List[float],
) -> plt.Figure:
    """Plot a horizontal bar chart of WER scores with color coding."""
    # Color thresholds: green=perfect, blue=good, orange=moderate, red=poor
    colors = []
    for s in scores:
        if s == 0.0:
            colors.append("seagreen")
        elif s < 0.20:
            colors.append("steelblue")
        elif s < 0.50:
            colors.append("darkorange")
        else:
            colors.append("firebrick")

    # Height scales with number of bars so labels stay readable
    fig, ax = plt.subplots(figsize=(10, max(3, len(labels) * 0.6)))
    y_pos = range(len(labels))
    bars = ax.barh(y_pos, [s * 100 for s in scores], color=colors, edgecolor="white")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.set_xlabel("WER (%)")
    ax.set_title("Word Error Rate per transcription")
    ax.invert_yaxis()
    ax.axvline(x=0, color="black", linewidth=0.8)
    ax.grid(axis="x", alpha=0.3)

    for bar, score in zip(bars, scores):
        ax.text(
            bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{score:.0%}", va="center", fontsize=9,
        )

    plt.tight_layout()
    return fig

## 3. Create Demo Data

We use synthetic reference/hypothesis pairs that mimic typical ASR errors on medical speech.

**Note:** Run Setup sections (Dependencies, Import, Functions) once before running any step cell. Each step cell loads demo data on its own.

In [ ]:
# Load synthetic reference/hypothesis pairs (no external files needed)
pairs = get_demo_pairs()

print(f"{len(pairs)} demo pairs loaded:\n")
for i, (ref, hyp) in enumerate(pairs, 1):
    print(f"Pair {i}:")
    print(f"  Reference:  {ref}")
    print(f"  Hypothesis: {hyp}\n")

## 4. Step 1 — Normalize Text

Before comparing, both texts are lowercased and stripped of punctuation so that WER measures **word accuracy**, not formatting.

In [ ]:
pairs = get_demo_pairs()
# Pair 0 differs only in case/punctuation — normalization makes them identical
ref, hyp = pairs[0]

print("BEFORE normalization:")
print(f"  Reference:  '{ref}'")
print(f"  Hypothesis: '{hyp}'")

# Apply lowercase + punctuation removal
ref_norm = normalize_text(ref)
hyp_norm = normalize_text(hyp)

print("\nAFTER normalization:")
print(f"  Reference:  '{ref_norm}'")
print(f"  Hypothesis: '{hyp_norm}'")
print(f"\nIdentical after normalization: {ref_norm == hyp_norm}")

## 5. Step 2 — Calculate WER for a Single Pair

We take a pair with actual errors and calculate WER step by step.

In [ ]:
pairs = get_demo_pairs()
# Pair 1 has two errors: "sharp"→"shark" (substitution) and "a" (insertion)
ref, hyp = pairs[1]

ref_norm = normalize_text(ref)
hyp_norm = normalize_text(hyp)

print(f"Reference (normalized):  '{ref_norm}'")
print(f"Hypothesis (normalized): '{hyp_norm}'")

# calculate_wer normalizes internally, so we pass the raw strings
score = calculate_wer(ref, hyp)
ref_words = ref_norm.split()

# N = number of words in the reference (denominator of WER formula)
print(f"\nReference word count (N): {len(ref_words)}")
print(f"WER = {score:.4f}  ({score:.0%})")
print(f"\nManual check:")
print(f"  Substitutions: 1 ('sharp' → 'shark')")
print(f"  Insertions:    1 ('a' added)")
print(f"  Deletions:     0")
print(f"  WER = (1 + 1 + 0) / {len(ref_words)} = 2/{len(ref_words)} = {2/len(ref_words):.4f}")

## 6. Step 3 — Interpret the WER Score

WER is a ratio: errors divided by reference length. Knowing the score tells you how reliable the transcription is.

In [ ]:
pairs = get_demo_pairs()

# Select pairs that showcase different error types for comparison
examples = [
    ("Perfect match", pairs[0]),      # only case/punctuation differences
    ("1 sub + 1 ins", pairs[1]),      # substitution + insertion
    ("1 deletion",    pairs[3]),      # missing word
    ("1 substitution", pairs[7]),     # wrong word
]

for label, (ref, hyp) in examples:
    score = calculate_wer(ref, hyp)
    # word accuracy = complement of WER (approximate, since WER can exceed 1.0)
    accuracy = (1 - score) * 100
    print(f"[{label}]")
    print(f"  Ref: {ref}")
    print(f"  Hyp: {hyp}")
    print(f"  WER: {score:.4f}  ({score:.0%})  →  word accuracy ≈ {accuracy:.0f}%\n")

## 7. Step 4 — Handle Edge Cases

The `calculate_wer` function handles situations where reference or hypothesis is empty.

In [ ]:
# These edge cases test the guard clauses in calculate_wer:
# empty ref → 1.0 (or 0.0 if both empty); empty hyp → all words deleted
edge_cases = [
    ("Empty reference, non-empty hypothesis", "", "some transcription"),
    ("Empty reference, empty hypothesis",     "", ""),
    ("Non-empty reference, empty hypothesis", "the patient reports pain", ""),
    ("Whitespace-only reference",             "   ", "hello"),
]

for label, ref, hyp in edge_cases:
    score = calculate_wer(ref, hyp)
    print(f"[{label}]")
    print(f"  Ref: '{ref}'  |  Hyp: '{hyp}'")
    print(f"  WER: {score:.4f}  ({score:.0%})\n")

## 8. Step 5 — Batch WER over Multiple Transcriptions

In practice, WER is computed over an entire dataset. We calculate individual and average WER.

In [ ]:
pairs = get_demo_pairs()
# Compute WER for every pair in one call
scores = batch_wer(pairs)

print(f"{'#':<4} {'WER':>8}  {'Reference (first 50 chars)'}")
print("-" * 70)
for i, ((ref, hyp), score) in enumerate(zip(pairs, scores), 1):
    print(f"{i:<4} {score:>7.0%}  {ref[:50]}")

# Summary statistics — typical reporting for ASR evaluation
avg = sum(scores) / len(scores)
perfect = sum(1 for s in scores if s == 0.0)
wer_1 = sum(1 for s in scores if s >= 1.0)  # WER=1.0 means total failure

print(f"\n{'='*70}")
print(f"Total files:   {len(scores)}")
print(f"Average WER:   {avg:.4f}  ({avg:.2%})")
print(f"Perfect (0%):  {perfect}")
print(f"WER >= 100%:   {wer_1}")

## 9. Visualize WER Distribution

A bar chart shows the WER for each transcription at a glance. Colors indicate quality:

- **Green** = perfect (0%)
- **Blue** = good (< 20%)
- **Orange** = moderate (20–50%)
- **Red** = poor (>= 50%)

In [ ]:
# Recompute scores so this cell is runnable independently
pairs = get_demo_pairs()
scores = batch_wer(pairs)
labels = [f"Pair {i}" for i in range(1, len(pairs) + 1)]

# Color-coded bar chart: green=0%, blue=<20%, orange=20-50%, red=>=50%
plot_wer_distribution(labels, scores)
plt.show()

## 10. Flow Summary

```
Reference text (ground truth)
    ↓  normalize_text()
Lowercased, no punctuation
    ↓
                              ┐
Hypothesis text (ASR output)  │
    ↓  normalize_text()       │  jiwer.wer()
Lowercased, no punctuation    │  aligns words,
    ↓                         │  counts S, I, D
                              ┘
    ↓
WER = (S + I + D) / N
    ↓
Score: 0.0 (perfect) to 1.0+ (many errors)
```

## Key Point

**WER measures word-level accuracy by counting substitutions, insertions, and deletions against a reference transcript.** Text is always normalized (lowercased, punctuation removed) before comparison so that WER reflects actual word accuracy, not formatting differences.

## Conclusion

### What You Learned

In this tutorial, you learned how to calculate and interpret Word Error Rate — the standard metric for evaluating speech recognition systems.

**Key concepts covered:**

1. **The WER Metric**
   - WER = (Substitutions + Insertions + Deletions) / Total reference words
   - 0% = perfect, higher = worse; can exceed 100% if many insertions
   - Standard metric used across ASR research and industry

2. **Edit Operations**
   - **Substitution**: wrong word in place of correct word
   - **Insertion**: extra word added by the system
   - **Deletion**: correct word missing from the output

3. **Text Normalization**
   - Lowercase all text before comparison
   - Remove punctuation so formatting differences don't inflate WER
   - This is standard practice in ASR evaluation

4. **Implementation**
   - The `jiwer` library computes WER via word-level alignment
   - Edge cases: empty reference returns 1.0 (or 0.0 if both empty)
   - Batch processing: compute WER per file, then average across the dataset

5. **Visualization**
   - Color-coded bar charts show quality at a glance
   - Average WER summarizes overall system performance

### Next Steps

You can now:
- Evaluate any ASR system by comparing its output against reference transcripts
- Use WER to compare different models or configurations
- Apply text normalization to ensure fair comparisons
- Visualize and report WER results across datasets
- Investigate high-WER transcriptions to understand failure modes